In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "NVDA", "TSLA"]

print("AVA-AI Version 2 — Advanced Features")
print("Raw data directory:", RAW_DIR.resolve())
print("Tickers:", ", ".join(tickers))

AVA-AI Version 2 — Advanced Features
Raw data directory: C:\Users\Aryan\Desktop\AVA-AI_MODEL-2\data\raw
Tickers: AAPL, AMZN, GOOGL, MSFT, NVDA, TSLA


In [2]:
stocks = {}

for ticker in tickers:
    file_path = RAW_DIR / f"{ticker}_daily.csv"
    df = pd.read_csv(file_path, parse_dates=["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    stocks[ticker] = df

print("Loaded stocks:")
for ticker, df in stocks.items():
    print(f"{ticker}: {len(df)} rows")

Loaded stocks:
AAPL: 500 rows
AMZN: 500 rows
GOOGL: 500 rows
MSFT: 500 rows
NVDA: 500 rows
TSLA: 500 rows


In [3]:
def add_v2_features(df):
    df = df.copy()

    # Basic returns
    df["return"] = df["close"].pct_change()
    df["return_lag1"] = df["return"].shift(1)
    df["return_lag2"] = df["return"].shift(2)
    df["return_lag3"] = df["return"].shift(3)
    df["return_lag5"] = df["return"].shift(5)
    df["return_lag10"] = df["return"].shift(10)

    # Moving averages
    df["MA5"] = df["close"].rolling(5).mean()
    df["MA20"] = df["close"].rolling(20).mean()
    df["MA50"] = df["close"].rolling(50).mean()

    # Price relative to moving averages
    df["price_vs_MA5"] = df["close"] / df["MA5"] - 1
    df["price_vs_MA20"] = df["close"] / df["MA20"] - 1
    df["price_vs_MA50"] = df["close"] / df["MA50"] - 1

    # Rolling returns
    df["rolling_return_5"] = df["return"].rolling(5).mean()
    df["rolling_return_10"] = df["return"].rolling(10).mean()
    df["rolling_return_20"] = df["return"].rolling(20).mean()

    # Volatility
    df["volatility_5"] = df["return"].rolling(5).std()
    df["volatility_10"] = df["return"].rolling(10).std()
    df["volatility_20"] = df["return"].rolling(20).std()

    # Volume
    df["volume_change"] = df["volume"].pct_change()
    df["volume_MA20"] = df["volume"].rolling(20).mean()
    df["volume_vs_MA20"] = df["volume"] / df["volume_MA20"] - 1

    # RSI
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_histogram"] = df["MACD"] - df["MACD_signal"]

    # Target: next-day return
    df["target_return"] = df["close"].shift(-1) / df["close"] - 1

    return df


for ticker in tickers:
    stocks[ticker] = add_v2_features(stocks[ticker])

print("V2 feature engineering complete.")
print("AAPL columns:", len(stocks["AAPL"].columns))
print(stocks["AAPL"].columns.tolist())

V2 feature engineering complete.
AAPL columns: 32
['datetime', 'open', 'high', 'low', 'close', 'volume', 'return', 'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5', 'return_lag10', 'MA5', 'MA20', 'MA50', 'price_vs_MA5', 'price_vs_MA20', 'price_vs_MA50', 'rolling_return_5', 'rolling_return_10', 'rolling_return_20', 'volatility_5', 'volatility_10', 'volatility_20', 'volume_change', 'volume_MA20', 'volume_vs_MA20', 'RSI_14', 'MACD', 'MACD_signal', 'MACD_histogram', 'target_return']


In [4]:
v2_features = [
    "close",
    "MA5",
    "MA20",
    "MA50",
    "price_vs_MA5",
    "price_vs_MA20",
    "price_vs_MA50",
    "volatility_5",
    "volatility_10",
    "volatility_20",
    "volume_change",
    "volume_vs_MA20",
    "return_lag1",
    "return_lag2",
    "return_lag3",
    "return_lag5",
    "return_lag10",
    "rolling_return_5",
    "rolling_return_10",
    "rolling_return_20",
    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_histogram",
]

print("V2 feature count:", len(v2_features))
print("Features:")
for feature in v2_features:
    print("-", feature)

V2 feature count: 24
Features:
- close
- MA5
- MA20
- MA50
- price_vs_MA5
- price_vs_MA20
- price_vs_MA50
- volatility_5
- volatility_10
- volatility_20
- volume_change
- volume_vs_MA20
- return_lag1
- return_lag2
- return_lag3
- return_lag5
- return_lag10
- rolling_return_5
- rolling_return_10
- rolling_return_20
- RSI_14
- MACD
- MACD_signal
- MACD_histogram


In [5]:
walk_splits = [
    (150, 150, 210),
    (210, 210, 270),
    (270, 270, 330),
    (330, 330, 390),
    (390, 390, 450),
]

prepared_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(subset=v2_features + ["target_return"]).copy()

    prepared_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_features)}"
    )

AAPL: 450 usable rows | features=24
AMZN: 450 usable rows | features=24
GOOGL: 450 usable rows | features=24
MSFT: 450 usable rows | features=24
NVDA: 450 usable rows | features=24
TSLA: 450 usable rows | features=24


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

v2_results = []

for ticker in tickers:
    df = prepared_data[ticker]

    X = df[v2_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(mean_absolute_error(y_test, predictions))
        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    v2_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

v2_results_df = pd.DataFrame(v2_results)

print(v2_results_df.to_string(index=False))

print("\nV2 overall:")
print("MAE :", v2_results_df["MAE"].mean())
print("RMSE:", v2_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.013948 0.018329
  AMZN 0.017560 0.023984
 GOOGL 0.019229 0.024829
  MSFT 0.014142 0.019471
  NVDA 0.019300 0.024581
  TSLA 0.025086 0.031960

V2 overall:
MAE : 0.01821080903151551
RMSE: 0.023858789032059863


In [ ]:
## Experiment 1 — Expanded Technical Features

V2 expanded the technical feature set from the Base Model by adding:

- Shorter moving averages
- Additional return lags
- Multiple volatility windows
- Volume-relative features
- Rolling return features
- MACD histogram
- Price-to-moving-average relationships

The Random Forest model was evaluated using the same walk-forward validation framework as the Base Model.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.013948 | 0.018329 |
| AMZN | 0.017560 | 0.023984 |
| GOOGL | 0.019229 | 0.024829 |
| MSFT | 0.014142 | 0.019471 |
| NVDA | 0.019300 | 0.024581 |
| TSLA | 0.025086 | 0.031960 |

Overall:

- MAE: **0.018211**
- RMSE: **0.023859**

Compared with the V1 Random Forest, the expanded feature set did not improve the overall average error.

The results were mixed across individual stocks, with improvements for AAPL, AMZN, GOOGL, NVDA and TSLA, while MSFT was slightly worse.

### Interpretation

Adding more technical indicators alone did not provide consistent evidence of improved predictive performance.

This experiment will therefore be treated as an exploratory V2 result rather than a replacement for the Base Model.

The next experiment will introduce **market-context features** so that the model can incorporate broader market movements rather than relying only on the individual stock's historical behavior.

In [7]:
market_file = RAW_DIR / "SP500_daily.csv"

print("Expected market data file:")
print(market_file.resolve())
print("Exists:", market_file.exists())

Expected market data file:
C:\Users\Aryan\Desktop\AVA-AI_MODEL-2\data\raw\SP500_daily.csv
Exists: False


In [8]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("TWELVE_DATA_API_KEY")

url = "https://api.twelvedata.com/time_series"

params = {
    "symbol": "SPX",
    "interval": "1day",
    "outputsize": 500,
    "apikey": api_key,
}

response = requests.get(url, params=params, timeout=30)
data = response.json()

print("API status:", data.get("status", "success"))
print("Returned rows:", len(data.get("values", [])))
print("Message:", data.get("message", "No error"))

API status: error
Returned rows: 0
Message: This symbol is available starting with the Grow or Venture plan. Consider upgrading now at https://twelvedata.com/pricing


In [9]:
import importlib.util

print("yfinance installed:", importlib.util.find_spec("yfinance") is not None)

yfinance installed: False


In [10]:
%pip install yfinance

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 2.8 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 3.3 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.1 MB 2.9 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.1 MB 2.7 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.1 MB 2.8 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.1 MB 2.8 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.1 MB 2.7 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.1 MB 2.8 MB/s eta 0:00:01
   ---------------------------------

In [11]:
import yfinance as yf

print("yfinance version:", yf.__version__)

yfinance version: 1.7.0


In [12]:
market_data = yf.download(
    "^GSPC",
    period="2y",
    interval="1d",
    auto_adjust=False,
    progress=False,
)

print("Rows:", len(market_data))
print("Columns:", market_data.columns.tolist())
print("Date range:", market_data.index.min(), "to", market_data.index.max())

Rows: 500
Columns: [('Adj Close', '^GSPC'), ('Close', '^GSPC'), ('High', '^GSPC'), ('Low', '^GSPC'), ('Open', '^GSPC'), ('Volume', '^GSPC')]
Date range: 2024-09-30 00:00:00 to 2026-09-28 00:00:00


In [13]:
market_df = market_data.copy()

market_df.columns = [
    "adj_close",
    "close",
    "high",
    "low",
    "open",
    "volume",
]

market_df = market_df.reset_index()
market_df = market_df.rename(columns={"Date": "datetime"})

market_df["datetime"] = pd.to_datetime(market_df["datetime"]).dt.tz_localize(None)

market_df = market_df.sort_values("datetime").reset_index(drop=True)

print(market_df.head())
print("\nRows:", len(market_df))
print("Missing values:", market_df.isna().sum().sum())
print("Date range:", market_df["datetime"].min(), "to", market_df["datetime"].max())

    datetime    adj_close        close         high          low         open  \
0 2024-09-30  5762.479980  5762.479980  5765.140137  5703.529785  5726.520020   
1 2024-10-01  5708.750000  5708.750000  5757.729980  5681.279785  5757.729980   
2 2024-10-02  5709.540039  5709.540039  5719.629883  5674.000000  5698.140137   
3 2024-10-03  5699.939941  5699.939941  5718.779785  5677.370117  5698.189941   
4 2024-10-04  5751.069824  5751.069824  5753.209961  5702.830078  5737.479980   

       volume  
0  4425730000  
1  4025180000  
2  3829050000  
3  3597450000  
4  3479400000  

Rows: 500
Missing values: 0
Date range: 2024-09-30 00:00:00 to 2026-09-28 00:00:00


In [14]:
market_df["market_return"] = market_df["close"].pct_change()
market_df["market_return_lag1"] = market_df["market_return"].shift(1)
market_df["market_return_lag5"] = market_df["market_return"].shift(5)

market_df["market_MA20"] = market_df["close"].rolling(20).mean()
market_df["market_MA50"] = market_df["close"].rolling(50).mean()

market_df["market_vs_MA20"] = (
    market_df["close"] / market_df["market_MA20"] - 1
)

market_df["market_vs_MA50"] = (
    market_df["close"] / market_df["market_MA50"] - 1
)

market_df["market_volatility_20"] = (
    market_df["market_return"].rolling(20).std()
)

market_features = [
    "market_return",
    "market_return_lag1",
    "market_return_lag5",
    "market_vs_MA20",
    "market_vs_MA50",
    "market_volatility_20",
]

print("Market features created:", len(market_features))
print(market_df[["datetime"] + market_features].tail())

Market features created: 6
      datetime  market_return  market_return_lag1  market_return_lag5  \
495 2026-09-22      -0.000008            0.014927           -0.004495   
496 2026-09-23      -0.007548           -0.000008           -0.004472   
497 2026-09-24      -0.000247           -0.007548            0.011381   
498 2026-09-25       0.005099           -0.000247            0.001668   
499 2026-09-28      -0.007369            0.005099            0.014927   

     market_vs_MA20  market_vs_MA50  market_volatility_20  
495        0.012471        0.018179              0.006690  
496        0.004641        0.010063              0.006911  
497        0.004207        0.009465              0.006911  
498        0.009245        0.014055              0.006814  
499        0.001974        0.005980              0.006997  


In [15]:
for ticker in tickers:
    stocks[ticker] = stocks[ticker].merge(
        market_df[["datetime"] + market_features],
        on="datetime",
        how="left",
    )

print("Market context merged.")

for ticker in tickers:
    print(
        f"{ticker}: {len(stocks[ticker])} rows | "
        f"market missing: {stocks[ticker][market_features].isna().any(axis=1).sum()}"
    )

Market context merged.
AAPL: 500 rows | market missing: 50
AMZN: 500 rows | market missing: 50
GOOGL: 500 rows | market missing: 50
MSFT: 500 rows | market missing: 50
NVDA: 500 rows | market missing: 50
TSLA: 500 rows | market missing: 50


In [16]:
v2_market_features = v2_features + market_features

prepared_market_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_market_features + ["target_return"]
    ).copy()

    prepared_market_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_market_features)}"
    )

print("\nTotal features:", len(v2_market_features))

AAPL: 449 usable rows | features=30
AMZN: 449 usable rows | features=30
GOOGL: 449 usable rows | features=30
MSFT: 449 usable rows | features=30
NVDA: 449 usable rows | features=30
TSLA: 449 usable rows | features=30

Total features: 30


In [17]:
market_results = []

for ticker in tickers:
    df = prepared_market_data[ticker]

    X = df[v2_market_features]
    y = df["target_return"]

    fold_mae = []
    fold_rmse = []

    for train_end, test_start, test_end in walk_splits:
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        fold_mae.append(
            mean_absolute_error(y_test, predictions)
        )

        fold_rmse.append(
            mean_squared_error(y_test, predictions) ** 0.5
        )

    market_results.append({
        "ticker": ticker,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
    })

market_results_df = pd.DataFrame(market_results)

print(market_results_df.to_string(index=False))

print("\nV2 Market Context overall:")
print("MAE :", market_results_df["MAE"].mean())
print("RMSE:", market_results_df["RMSE"].mean())

ticker      MAE     RMSE
  AAPL 0.012721 0.017134
  AMZN 0.017074 0.023561
 GOOGL 0.018118 0.023945
  MSFT 0.014176 0.019432
  NVDA 0.018600 0.023729
  TSLA 0.025093 0.031882

V2 Market Context overall:
MAE : 0.01763025904018936
RMSE: 0.023280504920948044


In [ ]:
## Experiment 2 — Market Context Features

V2 was extended with six S&P 500 market-context features:

- Market daily return
- Previous market return
- Five-day lagged market return
- S&P 500 price relative to its 20-day moving average
- S&P 500 price relative to its 50-day moving average
- 20-day market volatility

The market data was obtained independently and merged with each stock dataset by trading date.

### Results

| Ticker | MAE | RMSE |
|---|---:|---:|
| AAPL | 0.012721 | 0.017134 |
| AMZN | 0.017074 | 0.023561 |
| GOOGL | 0.018118 | 0.023945 |
| MSFT | 0.014176 | 0.019432 |
| NVDA | 0.018600 | 0.023729 |
| TSLA | 0.025093 | 0.031882 |

Overall:

- MAE: **0.017630**
- RMSE: **0.023281**

### Interpretation

Adding market-context information improved the expanded V2 technical-feature model.

However, the resulting overall error remained slightly higher than the original V1 Random Forest baseline.

The experiment therefore provides evidence that broad-market information contains useful contextual information for some stocks, but it does not yet establish a consistently superior predictive model.

Further V2 experiments will test additional sources of information while maintaining the same walk-forward validation framework.

In [18]:
market_df["volatility_regime"] = pd.qcut(
    market_df["market_volatility_20"],
    q=3,
    labels=["low", "medium", "high"],
)

print("Volatility regime counts:")
print(market_df["volatility_regime"].value_counts().sort_index())

Volatility regime counts:
volatility_regime
low       160
medium    160
high      160
Name: count, dtype: int64


In [19]:
market_df["volatility_regime_code"] = (
    market_df["volatility_regime"]
    .map({
        "low": 0,
        "medium": 1,
        "high": 2,
    })
)

print(
    market_df[
        ["datetime", "market_volatility_20", "volatility_regime", "volatility_regime_code"]
    ].tail(10)
)

print("\nMissing regime codes:", market_df["volatility_regime_code"].isna().sum())

      datetime  market_volatility_20 volatility_regime volatility_regime_code
490 2026-09-15              0.005709               low                      0
491 2026-09-16              0.005603               low                      0
492 2026-09-17              0.006215               low                      0
493 2026-09-18              0.005918               low                      0
494 2026-09-21              0.006736               low                      0
495 2026-09-22              0.006690               low                      0
496 2026-09-23              0.006911            medium                      1
497 2026-09-24              0.006911            medium                      1
498 2026-09-25              0.006814               low                      0
499 2026-09-28              0.006997            medium                      1

Missing regime codes: 20


In [20]:
regime_features = [
    "volatility_regime_code",
]

for ticker in tickers:
    stocks[ticker] = stocks[ticker].merge(
        market_df[["datetime"] + regime_features],
        on="datetime",
        how="left",
    )

print("Volatility regime merged.")

for ticker in tickers:
    print(
        f"{ticker}: {len(stocks[ticker])} rows | "
        f"missing regime: {stocks[ticker]['volatility_regime_code'].isna().sum()}"
    )

Volatility regime merged.
AAPL: 500 rows | missing regime: 21
AMZN: 500 rows | missing regime: 21
GOOGL: 500 rows | missing regime: 21
MSFT: 500 rows | missing regime: 21
NVDA: 500 rows | missing regime: 21
TSLA: 500 rows | missing regime: 21


In [21]:
v2_regime_features = v2_market_features + [
    "volatility_regime_code"
]

prepared_regime_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_regime_features + ["target_return"]
    ).copy()

    prepared_regime_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features)}"
    )

print("\nTotal features:", len(v2_regime_features))v2_regime_features = v2_market_features + [
    "volatility_regime_code"
]

prepared_regime_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_regime_features + ["target_return"]
    ).copy()

    prepared_regime_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features)}"
    )

print("\nTotal features:", len(v2_regime_features))

SyntaxError: invalid syntax (896614021.py, line 19)

In [22]:
v2_regime_features = v2_market_features + [
    "volatility_regime_code"
]

prepared_regime_data = {}

for ticker in tickers:
    df = stocks[ticker].dropna(
        subset=v2_regime_features + ["target_return"]
    ).copy()

    prepared_regime_data[ticker] = df

    print(
        f"{ticker}: {len(df)} usable rows | "
        f"features={len(v2_regime_features)}"
    )

print("\nTotal features:", len(v2_regime_features))

AAPL: 449 usable rows | features=31
AMZN: 449 usable rows | features=31
GOOGL: 449 usable rows | features=31
MSFT: 449 usable rows | features=31
NVDA: 449 usable rows | features=31
TSLA: 449 usable rows | features=31

Total features: 31
